In [ ]:
# === SECTION 1: IMPORTS ===
import sympy
import sympy.core
import sympy.core.symbol
import sympy.core.parameters
import os
import copy
import time
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib
import matplotlib.pyplot as plt

from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, roc_auc_score
)

matplotlib.rcParams['figure.dpi'] = 300

SEEDS = [42, 123, 777]
RESULTS = {}

def set_seed(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

In [ ]:
# === SECTION 2: DATA LOADING AND PREPROCESSING ===
DATA_PATH = "Dataset/merged_CSVs.csv"

df = pd.read_csv(DATA_PATH, low_memory=False)
print("Raw shape:", df.shape)

def build_binary_dataset(df):
    df = df.copy()
    activity = df["activity"].astype(str).str.lower().str.strip()
    y = np.where(activity.str.startswith("benign"), 0, 1).astype(np.int64)
    drop_cols = [c for c in ["flow_id", "timestamp", "src_ip", "dst_ip", "label", "activity"] if c in df.columns]
    X_df = df.drop(columns=drop_cols)
    for col in X_df.columns:
        X_df[col] = pd.to_numeric(X_df[col], errors='coerce')
    X_df = X_df.replace([np.inf, -np.inf], np.nan).fillna(0)
    return X_df.astype(np.float32).values, y

X_all, y_all = build_binary_dataset(df)
print("X shape:", X_all.shape)
print("y distribution:", np.unique(y_all, return_counts=True))

# Fixed global train/test split (seed=42 for reproducibility of split)
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X_all, y_all, test_size=0.2, stratify=y_all, random_state=42
)
scaler = StandardScaler()
X_train_full = scaler.fit_transform(X_train_full).astype(np.float32)
X_test = scaler.transform(X_test).astype(np.float32)

input_dim = X_train_full.shape[1]
print("Train:", X_train_full.shape, "Test:", X_test.shape)
print("Train distribution:", np.unique(y_train_full, return_counts=True))
print("Test  distribution:", np.unique(y_test, return_counts=True))

In [ ]:
# === SECTION 3: CLIENT PARTITIONING ===

def create_iid_clients(X, y, n_clients, seed):
    skf = StratifiedKFold(n_splits=n_clients, shuffle=True, random_state=seed)
    clients = [(X[idx], y[idx]) for _, idx in skf.split(X, y)]
    return clients

def create_dirichlet_clients(X, y, n_clients, alpha, seed):
    rng = np.random.default_rng(seed)
    classes = np.unique(y)
    client_indices = [[] for _ in range(n_clients)]
    for cls in classes:
        cls_idx = np.where(y == cls)[0]
        rng.shuffle(cls_idx)
        proportions = rng.dirichlet(alpha=np.full(n_clients, alpha))
        counts = (proportions * len(cls_idx)).astype(int)
        remainder = len(cls_idx) - counts.sum()
        for i in range(remainder):
            counts[i % n_clients] += 1
        start = 0
        for c, count in enumerate(counts):
            client_indices[c].extend(cls_idx[start:start + count].tolist())
            start += count
    clients = []
    n_dropped = 0
    for idx in client_indices:
        idx = np.array(idx, dtype=int)
        if len(idx) == 0:
            n_dropped += 1
            continue  # drop empty clients (happens with very low alpha)
        clients.append((X[idx], y[idx]))
    if n_dropped:
        print(f"  [Dirichlet] Dropped {n_dropped} empty client(s) (alpha too low).")
    return clients

def log_client_distributions(clients, label=""):
    print(f"\n--- Client distributions ({label}) ---")
    for i, (Xc, yc) in enumerate(clients):
        counts = np.bincount(yc, minlength=2)
        total = len(yc)
        if total == 0:
            print(f"  Client {i}: n=0 (empty — skipped)")
            continue
        print(f"  Client {i}: n={total}, benign={counts[0]} ({100*counts[0]/total:.1f}%), attack={counts[1]} ({100*counts[1]/total:.1f}%)")


In [ ]:
# === SECTION 4: MODEL DEFINITION ===

class MLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 200),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(200, 100),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(100, 50),
            nn.ReLU(),
            nn.Linear(50, 2)
        )

    def forward(self, x):
        return self.net(x)

print("Input dim:", input_dim)
print("Model params:", sum(p.numel() for p in MLP(input_dim).parameters()), "total")

In [ ]:
import sympy
import sympy.core
import sympy.core.symbol

# === SECTION 5: Defining FL ALGORITHMS (FedAvg, FedProx, DW-FedAvg) ===

import torch.backends.cudnn as cudnn
cudnn.benchmark = True  # optimize CUDA kernels for fixed input sizes

def pin_clients_to_gpu(clients):
    """Convert numpy client data to GPU tensors once. Reuse across rounds."""
    pinned = []
    for Xc, yc in clients:
        pinned.append((
            torch.tensor(Xc, dtype=torch.float32, device=device),
            torch.tensor(yc, dtype=torch.long, device=device)
        ))
    return pinned

def make_loader_gpu(X_gpu, y_gpu, batch_size=1024, shuffle=True):
    """DataLoader from already-on-GPU tensors. No .to(device) per batch."""
    from torch.utils.data import TensorDataset, DataLoader
    if len(X_gpu) == 0:
        raise ValueError("make_loader_gpu: empty client dataset (0 samples).")
    ds = TensorDataset(X_gpu, y_gpu)
    actual_batch = min(batch_size, len(X_gpu))
    return DataLoader(ds, batch_size=actual_batch, shuffle=shuffle,
                      drop_last=False, pin_memory=False) 

def make_loader(X, y, batch_size=1024, shuffle=True):
    """Fallback for numpy inputs (used in centralized baseline)."""
    ds = TensorDataset(torch.tensor(X, dtype=torch.float32),
                       torch.tensor(y, dtype=torch.long))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                      drop_last=False, pin_memory=True) 

def get_class_weights(y_gpu):
    """Accept GPU tensor or numpy. Returns weight tensor on device."""
    if isinstance(y_gpu, torch.Tensor):
        y_np = y_gpu.cpu().numpy().astype(np.int64)
    else:
        y_np = y_gpu.astype(np.int64)
    counts = np.bincount(y_np, minlength=2)
    total = counts.sum()
    weights = total / (2.0 * np.maximum(counts, 1))
    return torch.tensor(weights, dtype=torch.float32, device=device)

# ── FedAvg aggregation ───────────────────────────────────────────────────────

def fedavg_state_dict(local_states, weights):
    total_weight = float(sum(weights))
    new_state = copy.deepcopy(local_states[0])
    for key in new_state.keys():
        aggregated = local_states[0][key] * (weights[0] / total_weight)
        for state, w in zip(local_states[1:], weights[1:]):
            aggregated = aggregated + state[key] * (w / total_weight)
        new_state[key] = aggregated
    return new_state

# ── Local training – FedAvg ──────────────────────────────────────────────────

def train_local(model, X_gpu, y_gpu, epochs=1, batch_size=1024):
    """X_gpu, y_gpu are already on device."""
    model.train()
    criterion = nn.CrossEntropyLoss(weight=get_class_weights(y_gpu))
    optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
    loader = make_loader_gpu(X_gpu, y_gpu, batch_size=batch_size)
    for _ in range(epochs):
        for xb, yb in loader:
            optimizer.zero_grad()
            criterion(model(xb), yb).backward()
            optimizer.step()
    return model

# ── Local training – FedProx ─────────────────────────────────────────────────

def train_local_fedprox(model, X_gpu, y_gpu, global_state_dict, mu, epochs=1, batch_size=1024):
    model.train()
    criterion = nn.CrossEntropyLoss(weight=get_class_weights(y_gpu))
    optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
    global_params = [
        v.clone().detach().to(device)
        for v in global_state_dict.values()
        if v.dtype in (torch.float32, torch.float16, torch.float64)
    ]
    loader = make_loader_gpu(X_gpu, y_gpu, batch_size=batch_size)
    for _ in range(epochs):
        for xb, yb in loader:
            optimizer.zero_grad()
            ce_loss = criterion(model(xb), yb)
            prox_loss = (mu / 2) * sum(
                ((p - g) ** 2).sum()
                for p, g in zip(model.parameters(), global_params)
            )
            (ce_loss + prox_loss).backward()
            optimizer.step()
    return model


from sklearn.model_selection import train_test_split

def split_client_train_val(xc, yc, val_fraction=0.1, seed=42):
    indices = np.arange(len(yc))

    try:
        train_idx, val_idx = train_test_split(
            indices,
            test_size=val_fraction,
            stratify=yc,
            random_state=seed
        )
    except ValueError:
        # Handles clients containing only one class
        train_idx, val_idx = train_test_split(
            indices,
            test_size=val_fraction,
            random_state=seed
        )

    return (
        xc[train_idx],
        yc[train_idx],
        xc[val_idx],
        yc[val_idx]
    )

# ── DW-FedAvg local training ─────────────────────────────────────────────────

def train_local_dw(
    model,
    X_gpu,
    y_gpu,
    X_val_gpu,
    y_val_gpu,
    epochs=1,
    batch_size=1024
):
    model.train()

    criterion = nn.CrossEntropyLoss(
        weight=get_class_weights(y_gpu)
    )

    optimizer = optim.Adam(
        model.parameters(),
        lr=1e-3,
        weight_decay=1e-5
    )

    loader = make_loader_gpu(
        X_gpu,
        y_gpu,
        batch_size=batch_size
    )

    for _ in range(epochs):
        for xb, yb in loader:
            optimizer.zero_grad()

            outputs = model(xb)
            loss = criterion(outputs, yb)

            loss.backward()
            optimizer.step()

    model.eval()

    with torch.no_grad():
        outputs = model(X_val_gpu)
        preds = torch.argmax(outputs, dim=1)

    val_acc = accuracy_score(
        y_val_gpu.cpu().numpy(),
        preds.cpu().numpy()
    )

    return model, val_acc


# ── Entropy-Weighted FedAvg ──────────────────────────────────────────────────

def entropy_weight(y_gpu):
    """Class entropy of client data. Returns scalar in [0, 1]."""
    y_np = y_gpu.cpu().numpy()
    counts = np.bincount(y_np, minlength=2).astype(float)
    total = counts.sum()
    if total == 0:
        return 0.0
    p = counts / total
    p = p[p > 0]
    entropy = float(-np.sum(p * np.log2(p)))
    return entropy  # 0=single class, 1=balanced

def ew_fedavg_state_dict(local_states, sample_counts, y_clients_gpu):
    """Entropy-weighted FedAvg aggregation."""
    entropies = np.array([entropy_weight(yc) for yc in y_clients_gpu])
    raw_weights = np.array(sample_counts, dtype=float) * entropies
    if raw_weights.sum() < 1e-8:
        raw_weights = np.array(sample_counts, dtype=float)
    weights = (raw_weights / raw_weights.sum()).tolist()
    return fedavg_state_dict(local_states, weights)

# ── Coordinate-wise Median (Byzantine-robust) ────────────────────────────────

def median_aggregate(local_states):
    """Coordinate-wise median aggregation (Byzantine-robust)."""
    new_state = copy.deepcopy(local_states[0])
    for key in new_state.keys():
        stacked = torch.stack([s[key].float() for s in local_states], dim=0)
        new_state[key] = torch.median(stacked, dim=0).values
    return new_state

# ── run_federated ────────────────────────────────────────────────────────────

def run_federated(
    clients, X_test, y_test,
    rounds=20, local_epochs=5, batch_size=1024,
    algorithm="fedavg",
    mu=0.01,
    dw_alpha=0.2,
    verbose=True,
    convergence_f1=0.93,
    seed=42
):
    n_clients = len(clients)
    global_model = MLP(input_dim).to(device)

        if algorithm == "dw_fedavg":
        dw_clients = []
    
        for i, (Xc, yc) in enumerate(clients):
            X_tr, y_tr, X_val, y_val = split_client_train_val(
                Xc,
                yc,
                val_fraction=0.1,
                seed=seed + i
            )
    
            dw_clients.append(
                (X_tr, y_tr, X_val, y_val)
            )
    
        clients_gpu = [
            (
                torch.tensor(X_tr, dtype=torch.float32, device=device),
                torch.tensor(y_tr, dtype=torch.long, device=device),
                torch.tensor(X_val, dtype=torch.float32, device=device),
                torch.tensor(y_val, dtype=torch.long, device=device)
            )
            for X_tr, y_tr, X_val, y_val in dw_clients
        ]
    
    else:
        clients_gpu = pin_clients_to_gpu(clients)

    X_test_gpu = torch.tensor(X_test, dtype=torch.float32, device=device)

    beta = np.full(n_clients, 1.0 / n_clients)
    prev_val_accs = np.zeros(n_clients)

    model_bytes = sum(p.numel() * 4 for p in global_model.parameters())
    comm_bytes_per_round = 2 * n_clients * model_bytes

    history = []

    for r in range(rounds):
        round_start = time.time()
        local_states = []
        sample_counts = []
        curr_val_accs = np.zeros(n_clients)

        for i, client_data in enumerate(clients_gpu):
            local_model = copy.deepcopy(global_model)
        
            if algorithm == "dw_fedavg":
                Xc_gpu, yc_gpu, X_val_gpu, y_val_gpu = client_data
            else:
                Xc_gpu, yc_gpu = client_data
        
            if algorithm == "fedavg":
                local_model = train_local(
                    local_model,
                    Xc_gpu,
                    yc_gpu,
                    epochs=local_epochs,
                    batch_size=batch_size
                )
        
            elif algorithm == "fedprox":
                global_sd = {
                    k: v.clone()
                    for k, v in global_model.state_dict().items()
                }
        
                local_model = train_local_fedprox(
                    local_model,
                    Xc_gpu,
                    yc_gpu,
                    global_sd,
                    mu=mu,
                    epochs=local_epochs,
                    batch_size=batch_size
                )
        
            elif algorithm == "dw_fedavg":
                local_model, val_acc = train_local_dw(
                    local_model,
                    Xc_gpu,
                    yc_gpu,
                    X_val_gpu,
                    y_val_gpu,
                    epochs=local_epochs,
                    batch_size=batch_size
                )
        
                curr_val_accs[i] = val_acc
        
            elif algorithm == "ew_fedavg":
                local_model = train_local(
                    local_model,
                    Xc_gpu,
                    yc_gpu,
                    epochs=local_epochs,
                    batch_size=batch_size
                )
        
            elif algorithm == "median":
                local_model = train_local(
                    local_model,
                    Xc_gpu,
                    yc_gpu,
                    epochs=local_epochs,
                    batch_size=batch_size
                )
        
            elif algorithm == "fedprox_mu01":
                global_sd = {
                    k: v.clone()
                    for k, v in global_model.state_dict().items()
                }
        
                local_model = train_local_fedprox(
                    local_model,
                    Xc_gpu,
                    yc_gpu,
                    global_sd,
                    mu=0.1,
                    epochs=local_epochs,
                    batch_size=batch_size
                )
        
            else:
                raise ValueError(f"Unknown algorithm: {algorithm}")
        
            local_states.append({
                k: v.detach().clone()
                for k, v in local_model.state_dict().items()
            })
        
            sample_counts.append(len(Xc_gpu))

        # Aggregation
        if algorithm in ("fedavg", "fedprox"):
            new_state = fedavg_state_dict(local_states, sample_counts)
        elif algorithm == "dw_fedavg":
            if r > 0:
                improved = curr_val_accs >= prev_val_accs
                beta = np.where(improved, beta + beta * dw_alpha, beta - beta * dw_alpha)
                beta = np.clip(beta, 1e-8, None)
                beta = beta / beta.sum()

                beta = np.clip(beta, 0.05 / n_clients, 0.95 / n_clients * n_clients)
                beta = beta / beta.sum()  # renormalize again after clipping
            prev_val_accs = curr_val_accs.copy()
            new_state = fedavg_state_dict(local_states, beta.tolist())

        elif algorithm == "ew_fedavg":
            y_clients_gpu = [yc for _, yc in clients_gpu]
            new_state = ew_fedavg_state_dict(local_states, sample_counts, y_clients_gpu)
        elif algorithm == "median":
            new_state = median_aggregate(local_states)
        elif algorithm == "fedprox_mu01":
            new_state = fedavg_state_dict(local_states, sample_counts)

        global_model.load_state_dict(new_state)

        metrics = evaluate(global_model, X_test, y_test)
        round_time = time.time() - round_start

        history.append({
            **metrics,
            "round_time_sec": round_time,
            "cumulative_comm_bytes": (r + 1) * comm_bytes_per_round
        })

        if verbose:
            print(f"  Round {r+1:2d} | F1={metrics['f1']:.4f} | "
                  f"AUC={metrics['auc']:.4f} | FPR={metrics['fpr']:.4f} | t={round_time:.1f}s")

    return global_model, history


In [ ]:
# === SECTION 6: EVALUATION UTILITIES ===

def evaluate(model, X_test, y_test):
    model.eval()
    with torch.no_grad():
        X_t = torch.tensor(X_test, dtype=torch.float32).to(device)
        logits = model(X_t)
        probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
        preds = torch.argmax(logits, dim=1).cpu().numpy()
    acc = accuracy_score(y_test, preds)
    prec = precision_score(y_test, preds, zero_division=0)
    rec  = recall_score(y_test, preds, zero_division=0)
    f1   = f1_score(y_test, preds, zero_division=0)
    cm   = confusion_matrix(y_test, preds, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    fpr  = fp / (fp + tn + 1e-12)
    auc  = roc_auc_score(y_test, probs) if len(np.unique(y_test)) == 2 else np.nan
    return {"accuracy": acc, "precision": prec, "recall": rec,
            "f1": f1, "auc": auc, "fpr": fpr, "confusion_matrix": cm}

def final_metrics(history, convergence_f1=0.93):
    """Return final-round metrics and first convergence round."""
    last = history[-1]
    conv_round = None
    for i, h in enumerate(history):
        if h["f1"] >= convergence_f1:
            conv_round = i + 1
            break
    comm_to_conv = None
    if conv_round is not None:
        comm_to_conv = history[conv_round - 1]["cumulative_comm_bytes"]
    return {
        "acc":  last["accuracy"],
        "f1":   last["f1"],
        "auc":  last["auc"],
        "fpr":  last["fpr"],
        "prec": last["precision"],
        "rec":  last["recall"],
        "conv_round":       conv_round,
        "comm_to_conv":     comm_to_conv,
        "avg_round_time":   np.mean([h["round_time_sec"] for h in history]),
        "total_comm_bytes": last["cumulative_comm_bytes"],
    }

def aggregate_seeds(metrics_list):
    """Given list of metric dicts, return mean Â± std dict."""
    keys = [k for k in metrics_list[0].keys() if isinstance(metrics_list[0][k], (int, float, np.floating)) and metrics_list[0][k] is not None]
    out = {}
    for k in keys:
        vals = [m[k] for m in metrics_list if m.get(k) is not None]
        out[f"{k}_mean"] = np.mean(vals)
        out[f"{k}_std"]  = np.std(vals)
    return out

print("Evaluation utilities loaded.")

In [ ]:
# === SECTION 7: CENTRALIZED BASELINE ===
def run_centralized(X_train, y_train, X_test, y_test,
                    epochs=100, batch_size=1024):

    model = MLP(input_dim).to(device)
    loader = make_loader(X_train, y_train, batch_size=batch_size)

    criterion = nn.CrossEntropyLoss(
        weight=get_class_weights(y_train)
    )

    optimizer = optim.Adam(
        model.parameters(),
        lr=1e-3,
        weight_decay=1e-5
    )

    model.train()

    for _ in range(epochs):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)

            optimizer.zero_grad()
            criterion(model(xb), yb).backward()
            optimizer.step()

    return evaluate(model, X_test, y_test)
    
print("Centralized Baseline — Aggregated Results (HARDCODED):")
print(f"  Accuracy  : {RESULTS['centralized']['acc_mean']:.4f} ± {RESULTS['centralized']['acc_std']:.4f}")
print(f"  Precision : {RESULTS['centralized']['precision_mean']:.4f} ± {RESULTS['centralized']['precision_std']:.4f}")
print(f"  Recall    : {RESULTS['centralized']['recall_mean']:.4f} ± {RESULTS['centralized']['recall_std']:.4f}")
print(f"  F1        : {RESULTS['centralized']['f1_mean']:.4f} ± {RESULTS['centralized']['f1_std']:.4f}")
print(f"  AUC       : {RESULTS['centralized']['auc_mean']:.4f} ± {RESULTS['centralized']['auc_std']:.4f}")
print(f"  FPR       : {RESULTS['centralized']['fpr_mean']:.4f} ± {RESULTS['centralized']['fpr_std']:.4f}")


In [ ]:
# === SECTION 8: EXPERIMENT 1 — IID BENCHMARK ===
import sympy
import sympy.core
import sympy.core.symbol

# Methods: Centralized, FedAvg, FedProx(μ=0.01), DW-FedAvg
# Client counts: 4, 8, 16   |   Rounds: 20   |   Local epochs: 5

import time
exp_start_time = time.time()

EXP1_ROUNDS = 20
EXP1_EPOCHS = 5
EXP1_CLIENTS = [4, 8, 16]
EXP1_METHODS = [
    ("fedavg",   dict()),
    ("fedprox",  dict(mu=0.01)),
    ("dw_fedavg", dict()),
]

# history storage for plots (key = (method, n_clients))
exp1_histories = {}

for n_clients in EXP1_CLIENTS:
    for method_name, method_kwargs in EXP1_METHODS:
        key = f"exp1_{method_name}_iid_{n_clients}clients"
        print(f"{'='*60}")
        print(f"Exp1 | {method_name} | IID | n_clients={n_clients}")
        print(f"{'='*60}")
        seed_metrics = []
        histories_per_seed = []
        for s in SEEDS:
            print(f"  seed={s}")
            set_seed(s)
            clients = create_iid_clients(X_train_full, y_train_full, n_clients=n_clients, seed=s)
            log_client_distributions(clients, label=f"IID n={n_clients} seed={s}")
            _, history = run_federated(
                clients, X_test, y_test,
                rounds=EXP1_ROUNDS, local_epochs=EXP1_EPOCHS,
                algorithm=method_name, **method_kwargs
            )
            seed_metrics.append(final_metrics(history))
            histories_per_seed.append(history)
        RESULTS[key] = aggregate_seeds(seed_metrics)
        exp1_histories[(method_name, n_clients)] = histories_per_seed
        print(f"  => F1:   {RESULTS[key]['f1_mean']:.4f} ± {RESULTS[key]['f1_std']:.4f}")
        print(f"  => Acc:  {RESULTS[key]['acc_mean']:.4f} ± {RESULTS[key]['acc_std']:.4f}")
        print(f"  => Prec: {RESULTS[key]['prec_mean']:.4f} ± {RESULTS[key]['prec_std']:.4f}")
        print(f"  => Rec:  {RESULTS[key]['rec_mean']:.4f} ± {RESULTS[key]['rec_std']:.4f}")
        print(f"  => FPR:  {RESULTS[key]['fpr_mean']:.4f} ± {RESULTS[key]['fpr_std']:.4f}")

end_time = time.time()
print(f"Total execution time for Experiment 1: {end_time - start_time:.2f} seconds")


In [ ]:
# === SECTION 9: EXPERIMENT 2 â€” NON-IID BENCHMARK ===
import sympy
import sympy.core
import sympy.core.symbol

# Methods: FedAvg, FedProx(Î¼=0.01), DW-FedAvg
# n_clients=8, alpha âˆˆ {0.1, 0.5, 1.0}   |   Rounds: 20

import time
exp_start_time = time.time()

import time

start_time = time.time()


EXP2_ROUNDS = 20
EXP2_EPOCHS = 5
EXP2_CLIENTS = 8
EXP2_ALPHAS = [0.1, 0.5, 1.0]
EXP2_METHODS = [
    ("fedavg",    dict()),
    ("fedprox",   dict(mu=0.01)),
    ("dw_fedavg", dict()),
]

exp2_histories = {}

for alpha in EXP2_ALPHAS:
    for method_name, method_kwargs in EXP2_METHODS:
        key = f"exp2_{method_name}_alpha{alpha}"
        print(f"\n{'='*60}")
        print(f"Exp2 | {method_name} | Dirichlet Î±={alpha} | n_clients={EXP2_CLIENTS}")
        print(f"{'='*60}")
        seed_metrics = []
        histories_per_seed = []
        for s in SEEDS:
            print(f"  seed={s}")
            set_seed(s)
            clients = create_dirichlet_clients(X_train_full, y_train_full,
                                               n_clients=EXP2_CLIENTS, alpha=alpha, seed=s)
            log_client_distributions(clients, label=f"Dirichlet Î±={alpha} seed={s}")
            _, history = run_federated(
                clients, X_test, y_test,
                rounds=EXP2_ROUNDS, local_epochs=EXP2_EPOCHS,
                algorithm=method_name, **method_kwargs
            )
            seed_metrics.append(final_metrics(history))
            histories_per_seed.append(history)
        RESULTS[key] = aggregate_seeds(seed_metrics)
        exp2_histories[(method_name, alpha)] = histories_per_seed
        print(f"  => F1: {RESULTS[key]['f1_mean']:.4f} Â± {RESULTS[key]['f1_std']:.4f}")

end_time = time.time()
print(f"Total execution time for Experiment 1: {end_time - start_time:.2f} seconds")
exp_end_time = time.time()

In [ ]:
# === SECTION 10: NEW METHODS ON NON-IID ===
import sympy
import sympy.core
import sympy.core.symbol

# Methods: EW-FedAvg, Coordinate-wise Median, FedProx(μ=0.1)
# Setup: n_clients=8, Dirichlet α∈{0.1, 0.5, 1.0} + IID, 3 seeds, 20 rounds
# This is the paper's key contribution comparison.

import time

EXP5_ROUNDS  = 20
EXP5_EPOCHS  = 5
EXP5_ALPHAS  = [0.1, 0.5, 1.0]
EXP5_CLIENTS = 8

EXP5_METHODS = [
    ("ew_fedavg",      dict()),
    ("median",         dict()),
    ("fedprox_mu01",   dict()),
]

exp5_histories = {}
exp5_start = time.time()

# --- Non-IID runs ---
for alpha in EXP5_ALPHAS:
    for method_name, method_kwargs in EXP5_METHODS:
        key = f"exp5_{method_name}_alpha{alpha}"
        print(f"\n{'='*60}")
        print(f"Exp5 | {method_name} | Dirichlet α={alpha} | n_clients={EXP5_CLIENTS}")
        print(f"{'='*60}")
        seed_metrics = []
        histories_per_seed = []
        for s in SEEDS:
            print(f"  seed={s}")
            set_seed(s)
            clients = create_dirichlet_clients(
                X_train_full, y_train_full,
                n_clients=EXP5_CLIENTS, alpha=alpha, seed=s
            )
            _, history = run_federated(
                clients, X_test, y_test,
                rounds=EXP5_ROUNDS, local_epochs=EXP5_EPOCHS,
                algorithm=method_name, **method_kwargs
            )
            seed_metrics.append(final_metrics(history))
            histories_per_seed.append(history)
        RESULTS[key] = aggregate_seeds(seed_metrics)
        exp5_histories[(method_name, alpha)] = histories_per_seed
        r = RESULTS[key]
        print(f"  => F1:  {r['f1_mean']:.4f} ± {r['f1_std']:.4f}")
        print(f"  => AUC: {r['auc_mean']:.4f} ± {r['auc_std']:.4f}")
        print(f"  => FPR: {r['fpr_mean']:.4f} ± {r['fpr_std']:.4f}")

for method_name, method_kwargs in EXP5_METHODS:
    key = f"exp5_{method_name}_iid"
    print(f"\n{'='*60}")
    print(f"Exp5 | {method_name} | IID | n_clients={EXP5_CLIENTS}")
    print(f"{'='*60}")
    seed_metrics = []
    histories_per_seed = []
    for s in SEEDS:
        print(f"  seed={s}")
        set_seed(s)
        clients = create_iid_clients(
            X_train_full, y_train_full,
            n_clients=EXP5_CLIENTS, seed=s
        )
        _, history = run_federated(
            clients, X_test, y_test,
            rounds=EXP5_ROUNDS, local_epochs=EXP5_EPOCHS,
            algorithm=method_name, **method_kwargs
        )
        seed_metrics.append(final_metrics(history))
        histories_per_seed.append(history)
    RESULTS[key] = aggregate_seeds(seed_metrics)
    exp5_histories[(method_name, "iid")] = histories_per_seed
    r = RESULTS[key]
    print(f"  => F1:  {r['f1_mean']:.4f} ± {r['f1_std']:.4f}")

exp5_end = time.time()
print(f"\nTotal Exp5 time: {(exp5_end - exp5_start)/60:.1f} minutes")
